# Финальная Подготовка Модели (Clean Backtest)

В этом ноутбуке мы обучаем финальную модель, которую готовы отправить на реальную торговлю. Мы жестко отрезаем последние 2 месяца истории. Нейросеть учится на всем, кроме этих двух месяцев. Затем мы прогоняем её через отрезанный кусок и смотрим на результат. Если плюс — модель идет в продакшен.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, VecFrameStack
from stable_baselines3.common.monitor import Monitor
import wandb
from wandb.integration.sb3 import WandbCallback

from core.data.data_loader import load_crypto_data
from core.features.feature_generator import FeatureGenerator
from custom_envs.trading_env_v5 import TradingEnvV5
from agents.ppo_agent import create_ppo_agent, TradingMetricsCallback
from core.experiment_manager import get_experiment_paths

In [ ]:
# 1. Настройка эксперимента
exp_name = "ppo_gru_gated_hmm_2M"
model_path, norm_path, tb_log_dir = get_experiment_paths(exp_name)

print(f"Experiment Name: {exp_name}")
print(f"Model will be saved to: {model_path}")
print(f"Normalizer will be saved to: {norm_path}")
print(f"Tensorboard logs: {tb_log_dir}")

# Инициализация Weights & Biases
run = wandb.init(
    project="trading_rl",
    name=exp_name,
    sync_tensorboard=True,  
    monitor_gym=True,
    save_code=True,
)

## 2. Загрузка Данных и Train/Test Split

In [ ]:
df = load_crypto_data(symbol='BTCUSDT', start_date='2020-01-01', interval='4h', source='bybit_futures')

fg = FeatureGenerator()
data_features = fg.transform(df)

hmm_cols = [c for c in data_features.columns if 'hmm_regime' in c]
print(f"Data Features shape: {data_features.shape}")

# Отрезаем последние 2 месяца (6 свечей/день * 60 дней = 360 свечей)
TEST_SIZE = 360

train_df = data_features.iloc[:-TEST_SIZE].reset_index(drop=True)
test_df = data_features.iloc[-TEST_SIZE:].reset_index(drop=True)

print(f"Train size: {len(train_df)} свечей")
print(f"Test size: {len(test_df)} свечей")

## 3. Инициализация и Обучение на Train

In [ ]:
N_STACK = 10

train_env = TradingEnvV5(df=train_df, continuous_action=True, t_max=1000)
vec_train = VecFrameStack(DummyVecEnv([lambda: Monitor(train_env)]), n_stack=N_STACK)
vec_train = VecNormalize(vec_train, norm_obs=True, norm_reward=True, clip_obs=10., clip_reward=10.)

model = create_ppo_agent(
    vec_train, 
    extractor_type="gru", 
    num_hmm_states=len(hmm_cols), 
    n_stack=N_STACK,
    tensorboard_log=tb_log_dir
)

callbacks = [
    TradingMetricsCallback(),
    WandbCallback(
        gradient_save_freq=1000,
        model_save_path=model_path,
        verbose=0,
    )
]

print("Начинаем финальное обучение на Train выборке...")
model.learn(total_timesteps=2_000_000, progress_bar=False, callback=callbacks)

# Сохраняем модель и нормализатор по правильным путям
model.save(model_path)
vec_train.save(norm_path)
wandb.finish()
print(f"✅ Модель {exp_name} успешно обучена и сохранена.")

## 4. Валидация (Через Утилиту)
Этот блок использует нашу универсальную функцию валидации. Он подтягивает модель из правильного эксперимента.


In [ ]:
from core.evaluation.validation import run_validation

run_validation(
    data_features=data_features,
    model_path=model_path + ".zip",
    norm_path=norm_path,
    use_frame_stack=True,
    n_stack=10,
    test_size=360,
    random_start=False  # Для продакшена мы жестко смотрим на последний OOS участок
)